# Locality with numba — a hand loop sees what numpy hides.  ~4 min

numpy stores a matrix **row by row** (row-major). Fill one loop, run, read the two times.

In [ ]:
import numpy as np, time
from numba import njit

A = np.random.rand(4000, 4000)     # A[i, j] and A[i, j+1] are neighbours in memory

def best(f, r=5):
    f(A)                            # numba compiles on the 1st call
    ts = []
    for _ in range(r):
        t0 = time.perf_counter(); f(A); ts.append(time.perf_counter() - t0)
    return min(ts) * 1e3            # ms

In [ ]:
@njit
def sum_against(A):                 # i inner: each step jumps a whole row → strided
    s = 0.0
    for j in range(A.shape[1]):
        for i in range(A.shape[0]):
            s += A[i, j]
    return s

**Your turn.** Same sum, but walk **with the grain**: numpy is row-major, so make `j` the **inner** loop.

In [ ]:
@njit
def sum_with(A):
    s = 0.0
    ## SOLUTION: i outer, j inner — for i in range(A.shape[0]): for j in range(A.shape[1]) ##
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            s += A[i, j]
    ## END ##
    return s

In [ ]:
assert abs(sum_against(A) - sum_with(A)) < 1e-6
print("against the grain :", f"{best(sum_against):.0f} ms")
print("with the grain    :", f"{best(sum_with):.0f} ms")
print("→ ~10× on the same data — the inner loop should run along contiguous memory.")

### What you just saw

numpy is **row-major**, so the inner loop must run **along a row**. Julia is the **opposite** — **column-major** — so there the inner loop runs **down a column**. Same lesson, mirrored (`4_locality_julia.jl`).

Plain `np.sum` **hides** this: it reorders the iteration into memory order for you. The moment you write the loop yourself — numba here, or Julia — the memory hierarchy sends the bill.